# X-Ray Red: one-step FGSM challenge

Your goal is to change a classifier's predicted class for an approved chest X-ray with **one FGSM step**. The submitted image must remain visually close to the original: its measured pixel-space L∞ distance must be at most **0.02**. This is a model-robustness exercise, not a medical diagnosis.

Complete the TODO in section 2. Your local prediction is a useful check, but only the protected evaluator in section 3 can award the X-Ray Red flag. A rejected image earns no flag.


## 1. Choose an approved X-ray

Run the next cell to see the approved images. Your workspace login determines your Arena identity; the source ID identifies which original image the evaluator will compare with your PNG.


In [ ]:
import red_blue_arena as arena
from IPython.display import display, HTML, Image as DisplayImage

identity = arena.identity()
assert identity['role'] == 'red', 'This workspace is not assigned to Red.'
source_response = arena.get_sources()
sources = source_response['sources']
print(f"Approved images: {len(sources)}; maximum measured L∞: {source_response['max_linf']}")
display(arena.source_gallery(sources))


In [ ]:
# Choose an ID from the gallery. The first approved image is the default.
source_id = str(sources[0]['source_id'])
source_info = next((item for item in sources if str(item['source_id']) == source_id), None)
if source_info is None:
    raise ValueError(f'Source ID {source_id!r} is not approved.')
clean_path = arena.download_source(source_id)
clean_prediction = arena.predict_image(clean_path)
display(HTML(arena.source_info_card_html(source_info, clean_prediction)))
display(DisplayImage(filename=str(clean_path)))


## 2. TODO: create one FGSM adversarial image

FGSM increases the model's classification loss in the direction of the **sign of the input gradient**. Fill the three `None` values in the next cell: compute cross-entropy loss for the original class, differentiate it with respect to the image, then make **one** step and clamp pixel values to `[0, 1]`.

Use `epsilon = 5/255`: one extra 8-bit intensity level would exceed the evaluator's 0.02 limit. The model takes a 256×256 RGB tensor even though the source is an X-ray. Do not call `arena.make_adversarial_candidate()` for this TODO; implement the step yourself.


In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

model = arena.load_standard_model()
device = next(model.parameters()).device
with Image.open(clean_path) as opened:
    clean_pixels = np.asarray(opened.convert('RGB'), dtype=np.float32).copy() / 255.0
clean_tensor = torch.from_numpy(clean_pixels.transpose(2, 0, 1)).unsqueeze(0).to(device)
true_label = torch.tensor([int(source_info['class_index'])], device=device)
epsilon = 5 / 255
attack_input = clean_tensor.detach().clone().requires_grad_(True)

# TODO 1: Compute cross-entropy loss from model(attack_input) and true_label.
loss = None
# TODO 2: Differentiate that loss with respect to attack_input.
gradient = None
# TODO 3: Take ONE step in the sign of the gradient and clamp to [0, 1].
candidate_tensor = None

if loss is None or gradient is None or candidate_tensor is None:
    raise NotImplementedError('Complete all three FGSM TODO lines before running this cell.')
if not isinstance(candidate_tensor, torch.Tensor) or candidate_tensor.shape != clean_tensor.shape:
    raise ValueError('candidate_tensor must be a tensor with the same shape as clean_tensor.')
candidate_pixels = candidate_tensor.detach().clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
candidate_path = Path('/workspace/xray_red') / f'fgsm_source_{source_id}.png'
Image.fromarray(np.rint(candidate_pixels * 255).astype(np.uint8), mode='RGB').save(candidate_path)
candidate_prediction = arena.predict_image(candidate_path)
display(HTML(arena.prediction_card_html(clean_prediction, candidate_prediction)))
display(DisplayImage(filename=str(candidate_path)))


In [ ]:
with Image.open(clean_path) as original, Image.open(candidate_path) as changed:
    original_pixels = np.asarray(original.convert('RGB'), dtype=np.int16)
    changed_pixels = np.asarray(changed.convert('RGB'), dtype=np.int16)
measured_linf = float(np.abs(changed_pixels - original_pixels).max() / 255.0)
local_flip = candidate_prediction['class_index'] != clean_prediction['class_index']
print(f"Local prediction: {clean_prediction['class_label']} → {candidate_prediction['class_label']}")
print(f"Class changed locally: {local_flip}; measured PNG L∞: {measured_linf:.6f} / {source_response['max_linf']:.6f}")
display(arena.compare_images(clean_path, candidate_path))
if not local_flip or measured_linf > source_response['max_linf']:
    print('This local check did not pass. Revise your FGSM step or choose another approved source before submitting.')


## 3. Submit to the protected evaluator

The evaluator independently checks the original image, the submitted PNG's measured L∞ distance, and whether the model's top-1 class changed. A local class change is not a flag; the protected result below is authoritative. If a previous attack is still awaiting a response, the Arena will not accept a new one until that round is finished.


In [ ]:
import os
from html import escape

try:
    submission = arena.submit_red_attack(
        candidate_path, source_id, attack_name='FGSM',
        epsilon=epsilon, step_size=epsilon, iterations=1,
    )
except RuntimeError as exc:
    print('The protected evaluator did not accept this submission:', exc)
else:
    if submission.get('submitted') and submission.get('flag'):
        display(HTML(arena.attack_result_card_html(submission)))
        print('Red succeeded. Submit the flag above on the X-Ray Red challenge page in CTFd.')
        ctfd_url = os.getenv('CTFD_PUBLIC_URL', 'http://localhost:8001').rstrip('/')
        blue_url = f'{ctfd_url}/workspace-launch?notebook=xray_blue%2Fblue_team.ipynb'
        display(HTML(
            '<p><strong>Next: Blue Team.</strong> After submitting the Red flag in CTFd, '
            f'<a href="{escape(blue_url, quote=True)}">open the Blue notebook</a>'
            ' to defend this image.</p>'
        ))
    else:
        print('No Red flag was awarded for this submission.')
